# ED Pathway Trainer v3.3 (Self-Contained)
End-to-end: simulate → split (min per-class) → train → calibrate → thresholds (cost + ops) → CM/report → KPI → Policy/Bridge + audit anchor.
Artifacts go under `artifacts/`.

In [ ]:

# --- Imports, seeds, helpers ---
import os, json, math, random, textwrap, warnings, base64, io
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, log_loss

warnings.filterwarnings("ignore", category=RuntimeWarning, module="pandas.io.formats.format")

SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

ARTS = Path("artifacts"); ARTS.mkdir(parents=True, exist_ok=True)
PKG = Path("ed_core"); PKG.mkdir(parents=True, exist_ok=True)

ACTIONS = ["NO_OP","ORDER_ECG","PERFORM_FAST","ORDER_LABS","ORDER_XR","ORDER_CT","REQUEST_CONSULT","REQUEST_BED"]
X_COLS = ['minute_of_day','cap_stale','ems','consult_delay_min',
          'syn_chest_pain','syn_polytrauma','syn_neuro_deficit','syn_other',
          'ecg_hint','fast_hint','ct_hint','ems_prealert','risk_score']

def save_json(path, obj):
    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    with open(p, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)

def write_text(path, content):
    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(textwrap.dedent(content).lstrip("\n"), encoding="utf-8")

print("num_actions:", len(ACTIONS), "| num_features:", len(X_COLS))


In [ ]:

# --- Synthetic generator with scenario recipes & IM handoff packet coverage ---
def gen_day(n=300, day_index=0):
    # Generate a day of ED events with features and action labels.
    rows = []
    start_min = day_index*1440
    scenarios = [
        ("ACS",         0.10),
        ("Dyspnea",     0.12),
        ("Stroke",      0.08),
        ("Sepsis",      0.08),
        ("Abdomen",     0.10),
        ("DKA",         0.05),
        ("Arrhythmia",  0.08),
        ("Other",       0.39),
    ]

    for i in range(n):
        m = start_min + np.random.randint(0, 1440)
        minute_of_day = m % 1440
        night = (minute_of_day>=1200 or minute_of_day<360)
        cap_stale = np.random.rand() < (0.25 if night else 0.15)
        ems = np.random.rand() < 0.35
        consult_delay_min = max(0, np.random.gamma(2.0, 12.0) * (1.2 if night else 1.0))

        syn_chest_pain = syn_polytrauma = syn_neuro_deficit = 0
        s = random.choices([x[0] for x in scenarios], weights=[x[1] for x in scenarios])[0]
        if s=="ACS": syn_chest_pain=1
        if s=="Stroke": syn_neuro_deficit=1
        if s=="Other" and np.random.rand()<0.05: syn_polytrauma=1

        syn_other = int(not (syn_chest_pain or syn_polytrauma or syn_neuro_deficit))

        ecg_hint = 1 if syn_chest_pain or np.random.rand()<0.2 else 0
        fast_hint = 1 if syn_neuro_deficit or np.random.rand()<0.05 else 0
        ct_hint = 1 if (syn_neuro_deficit or syn_polytrauma or np.random.rand()<0.1) else 0
        ems_prealert = int(ems and (syn_neuro_deficit or syn_polytrauma or syn_chest_pain) and np.random.rand()<0.6)

        risk_score = np.clip(np.random.normal(0.5, 0.2) + 0.2*ems_prealert + 0.1*int(night), 0, 1)

        base = {
            "NO_OP": 0.40,
            "ORDER_ECG": 0.15 + 0.5*syn_chest_pain,
            "PERFORM_FAST": 0.05 + 0.7*syn_neuro_deficit,
            "ORDER_LABS": 0.20 + 0.15*(syn_chest_pain or syn_neuro_deficit),
            "ORDER_XR": 0.08 + 0.25*(1 if s=='Dyspnea' else 0),
            "ORDER_CT": 0.05 + 0.35*(ct_hint),
            "REQUEST_CONSULT": 0.03 + 0.20*(syn_neuro_deficit or syn_chest_pain),
            "REQUEST_BED": 0.02 + 0.05*(risk_score>0.8),
        }
        tot = sum(base.values()); probs = {k: v/tot for k,v in base.items()}
        action = random.choices(list(probs.keys()), weights=list(probs.values()))[0]

        im_packet_complete = bool(np.random.rand()<0.7)
        trauma_cleared = bool(np.random.rand()<0.8)
        capacity_fresh_im = bool(np.random.rand()<0.85)
        reason_code_im = random.choice(["syncope","arrhythmia","electrolyte","glycemic","infection","ACS"])
        if action in ("REQUEST_CONSULT","REQUEST_BED"):
            if not (im_packet_complete and trauma_cleared and capacity_fresh_im):
                action = "NO_OP"

        rows.append({
            "minute_of_day": minute_of_day, "cap_stale": int(cap_stale), "ems": int(ems),
            "consult_delay_min": float(consult_delay_min),
            "syn_chest_pain": int(syn_chest_pain), "syn_polytrauma": int(syn_polytrauma),
            "syn_neuro_deficit": int(syn_neuro_deficit), "syn_other": int(syn_other),
            "ecg_hint": int(ecg_hint), "fast_hint": int(fast_hint), "ct_hint": int(ct_hint),
            "ems_prealert": int(ems_prealert), "risk_score": float(risk_score),
            "label": action
        })
    return pd.DataFrame(rows)

def gen_dataset(days=5, per_day=350):
    dfs = [gen_day(per_day, d) for d in range(days)]
    df = pd.concat(dfs, ignore_index=True)
    need_min = {"ORDER_CT": 25, "REQUEST_CONSULT": 30, "REQUEST_BED": 20, "ORDER_XR": 25, "PERFORM_FAST": 25}
    for act, k in need_min.items():
        have = (df["label"]==act).sum()
        if have < k:
            add = []
            for _ in range(k - have):
                row = gen_day(1).iloc[0].to_dict()
                row["label"] = act
                if act=="REQUEST_BED": row["risk_score"]=0.95
                if act=="ORDER_CT": row["ct_hint"]=1
                if act=="PERFORM_FAST": row["fast_hint"]=1; row["syn_neuro_deficit"]=1
                if act=="ORDER_XR": row["syn_other"]=1
                if act=="REQUEST_CONSULT": row["ecg_hint"]=1 or row["fast_hint"]=1
                add.append(row)
            df = pd.concat([df, pd.DataFrame(add)], ignore_index=True)
    return df

df = gen_dataset(days=5, per_day=350)
print("Class counts:", df["label"].value_counts().to_dict())


In [ ]:

# --- Train/Val split with guaranteed min per-class in validation ---
from collections import Counter

def split_with_min_per_class(df, label_col="label", val_size=0.17, min_val_per_class=8, seed=1337):
    rng = np.random.default_rng(seed)
    idx = np.arange(len(df))
    rng.shuffle(idx)
    n_val = int(len(df)*val_size)
    val_idx = set(idx[:n_val].tolist())
    counts = Counter(df.loc[list(val_idx), label_col])
    for cls in df[label_col].unique():
        need = max(0, min_val_per_class - counts.get(cls, 0))
        if need>0:
            cand = df.index[(df[label_col]==cls) & (~df.index.isin(val_idx))].tolist()
            take = cand[:need]
            for j in take:
                k = next(iter(val_idx))
                val_idx.remove(k); val_idx.add(j)
            counts[cls] = counts.get(cls, 0) + len(take)
    train_idx = [i for i in df.index if i not in val_idx]
    return df.loc[train_idx].reset_index(drop=True), df.loc[list(val_idx)].reset_index(drop=True)

train_df, val_df = split_with_min_per_class(df, min_val_per_class=8)
print("Train/Val sizes:", len(train_df), len(val_df))
print("Val per-class:", val_df["label"].value_counts().to_dict())


In [ ]:

# --- Torch dataset/model/training ---
class TabDS(torch.utils.data.Dataset):
    def __init__(self, df):
        self.X = df[['minute_of_day','cap_stale','ems','consult_delay_min',
                     'syn_chest_pain','syn_polytrauma','syn_neuro_deficit','syn_other',
                     'ecg_hint','fast_hint','ct_hint','ems_prealert','risk_score']].astype(float).values.astype(np.float32)
        self.y = df['label'].map({k:i for i,k in enumerate(["NO_OP","ORDER_ECG","PERFORM_FAST","ORDER_LABS","ORDER_XR","ORDER_CT","REQUEST_CONSULT","REQUEST_BED"])}).values.astype(np.int64)
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return self.X[i][None, :], self.y[i]

ACTIONS = ["NO_OP","ORDER_ECG","PERFORM_FAST","ORDER_LABS","ORDER_XR","ORDER_CT","REQUEST_CONSULT","REQUEST_BED"]
X_COLS = ['minute_of_day','cap_stale','ems','consult_delay_min',
          'syn_chest_pain','syn_polytrauma','syn_neuro_deficit','syn_other',
          'ecg_hint','fast_hint','ct_hint','ems_prealert','risk_score']

train_ds, val_ds = TabDS(train_df), TabDS(val_df)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)

device = "cpu"
class GRUHead(nn.Module):
    def __init__(self, in_features, hidden=128, num_actions=8, p=0.3):
        super().__init__()
        self.gru = nn.GRU(in_features, hidden, batch_first=True)
        self.dropout = nn.Dropout(p)
        self.fc_multi = nn.Linear(hidden, num_actions)
    def forward(self, x):
        out, _ = self.gru(x)
        h = out[:, -1, :]
        h = self.dropout(h)
        return self.fc_multi(h)

model = GRUHead(in_features=len(X_COLS), hidden=128, num_actions=len(ACTIONS)).to(device)
assert model.fc_multi.out_features == len(ACTIONS)

opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()

def run_epoch(dl, train=True):
    model.train(mode=train)
    tot, n = 0.0, 0
    for xb, yb in dl:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = crit(logits, yb)
        if train:
            opt.zero_grad(); loss.backward(); opt.step()
        tot += float(loss.item())*len(yb); n += len(yb)
    return tot/n

for ep in range(1, 26):
    tr = run_epoch(train_loader, True)
    va = run_epoch(val_loader, False)
    if ep%1==0:
        print(f"Epoch {ep:02d}: train={tr:.4f} val={va:.4f}")

# Collect validation probs/labels
model.eval()
all_probs, all_true = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        logits = model(xb.to(device))
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        all_probs.append(probs); all_true.append(yb.numpy())
probs_val = np.concatenate(all_probs, 0)
y_true_val = np.concatenate(all_true, 0)
np.savez(Path('artifacts')/'val_cache.npz', probs=probs_val, y_true=y_true_val, action_names=np.array(ACTIONS, dtype=object))
print("Saved artifacts/val_cache.npz:", probs_val.shape, y_true_val.shape)


In [ ]:

# --- Temperature scaling (global) ---
from sklearn.metrics import log_loss
Ts = np.linspace(0.5, 2.0, 16)
bestT, bestNLL = 1.0, 1e9
for T in Ts:
    scaled = probs_val**(1.0/T); scaled = scaled / scaled.sum(axis=1, keepdims=True)
    nll = log_loss(y_true_val, scaled, labels=list(range(len(["NO_OP","ORDER_ECG","PERFORM_FAST","ORDER_LABS","ORDER_XR","ORDER_CT","REQUEST_CONSULT","REQUEST_BED"]))))
    if nll < bestNLL:
        bestNLL, bestT = nll, float(T)
with open(Path('artifacts')/'calibration.json','w') as f:
    json.dump({"temperature": bestT, "val_nll": bestNLL}, f, indent=2)
print(f"[Calib] Best temperature T={bestT:.3f}")


In [ ]:

# --- Per-class PR/Cost thresholds ---
ACTIONS = ["NO_OP","ORDER_ECG","PERFORM_FAST","ORDER_LABS","ORDER_XR","ORDER_CT","REQUEST_CONSULT","REQUEST_BED"]
def pick_tau(y_true_bin, p, fp_cost=1.0, fn_cost=5.0):
    taus = np.linspace(0,1,101)
    best = None
    for t in taus:
        yp = (p>=t).astype(int)
        tp = int(((yp==1)&(y_true_bin==1)).sum())
        fp = int(((yp==1)&(y_true_bin==0)).sum())
        fn = int(((yp==0)&(y_true_bin==1)).sum())
        prec = tp/(tp+fp+1e-9); rec = tp/(tp+fn+1e-9)
        f1 = 2*prec*rec/(prec+rec+1e-9)
        cost = fp_cost*fp + fn_cost*fn
        if (best is None) or (cost < best["cost"]) or (cost==best["cost"] and rec>best["recall"]):
            best = {"tau": float(t), "precision": float(prec), "recall": float(rec), "f1": float(f1), "cost": int(cost), "tp": tp, "fp": fp, "fn": fn}
    return best

thresholds = {}
for ci, name in enumerate(ACTIONS):
    yb = (y_true_val==ci).astype(int)
    thresholds[name] = pick_tau(yb, probs_val[:,ci])
with open(Path('artifacts')/'thresholds.json','w') as f:
    json.dump(thresholds, f, indent=2)
with open(Path('artifacts')/'per_class_thresholds_cost.json','w') as f:
    json.dump(thresholds, f, indent=2)
print("Wrote thresholds to artifacts/thresholds.json (+ legacy file)")

# --- Ops-aware thresholds (FP/hour budgets) ---
def choose_tau_under_fp_budget(y_true_bin, p, hours, fp_budget_per_hour=1.0):
    taus = np.linspace(0,1,101)
    best = None
    for t in taus:
        yp = (p>=t).astype(int)
        fp = int(((yp==1)&(y_true_bin==0)).sum())
        tp = int(((yp==1)&(y_true_bin==1)).sum())
        fn = int(((yp==0)&(y_true_bin==1)).sum())
        fph = fp / max(1e-9, hours)
        if fph <= fp_budget_per_hour:
            rec = tp/(tp+fn+1e-9); prec = tp/(tp+fp+1e-9)
            if (best is None) or (rec > best["recall"]) or (rec==best["recall"] and prec>best["precision"]):
                best = {"tau": float(t), "fp_per_hour": float(fph), "precision": float(prec), "recall": float(rec), "tp": tp, "fp": fp, "fn": fn}
    return best

hours = 24.0
notes = {"assumption_hours": hours, "comment": "synthetic run default"}
ops = {}
fp_budgets = {"REQUEST_CONSULT": 1.0, "REQUEST_BED": 0.5}
for ci, name in enumerate(ACTIONS):
    yb = (y_true_val==ci).astype(int); p = probs_val[:,ci]
    B = fp_budgets.get(name, 2.0)
    sel = choose_tau_under_fp_budget(yb, p, hours, fp_budget_per_hour=B)
    if sel is None:
        t = thresholds[name]
        sel = {"tau": t["tau"], "fp_per_hour": None, "precision": t["precision"], "recall": t["recall"], "tp": t["tp"], "fp": t["fp"], "fn": t["fn"], "fallback":"cost"}
    ops[name] = sel
with open(Path('artifacts')/'thresholds_ops.json','w') as f:
    json.dump(ops, f, indent=2)
with open(Path('artifacts')/'thresholds_ops_notes.json','w') as f:
    json.dump(notes, f, indent=2)
print("Wrote artifacts/thresholds_ops.json and thresholds_ops_notes.json")


In [ ]:

# --- Robust report + confusion matrix (force all labels) ---
ACTIONS = ["NO_OP","ORDER_ECG","PERFORM_FAST","ORDER_LABS","ORDER_XR","ORDER_CT","REQUEST_CONSULT","REQUEST_BED"]
labels_all = list(range(len(ACTIONS)))
y_pred_val = probs_val.argmax(axis=1)
rep = classification_report(y_true_val, y_pred_val, labels=labels_all, target_names=ACTIONS, zero_division=0, output_dict=True)
with open(Path('artifacts')/'classification_report.json','w') as f:
    json.dump(rep, f, indent=2)

cm = confusion_matrix(y_true_val, y_pred_val, labels=labels_all)
fig, ax = plt.subplots(figsize=(6,6))
im = ax.imshow(cm, interpolation='nearest')
ax.set_title("Confusion Matrix (Val)")
ax.set_xticks(np.arange(len(ACTIONS))); ax.set_yticks(np.arange(len(ACTIONS)))
ax.set_xticklabels(ACTIONS, rotation=45, ha="right"); ax.set_yticklabels(ACTIONS)
for i in range(len(ACTIONS)):
    for j in range(len(ACTIONS)):
        ax.text(j, i, int(cm[i,j]), ha="center", va="center", fontsize=8)
fig.tight_layout(); fig.savefig(Path('artifacts')/'cm_overall_v5.png', dpi=160); plt.close(fig)
print("Saved artifacts/cm_overall_v5.png\nSaved artifacts/classification_report.json")


In [ ]:

# --- Policy (pure functions) + Bridge (idempotent) ---
import py_compile
policy_py = r'''
ALLOWED_IM = {"syncope","arrhythmia","electrolyte","glycemic","infection","ACS"}

def require_approval(action_code: str) -> bool:
    return action_code in {"ORDER_CT","ORDER_XR","ORDER_LABS","ORDER_ECG","PERFORM_FAST",
                           "REQUEST_CONSULT","REQUEST_BED"}

def can_page_im(state: dict) -> bool:
    return bool(state.get("trauma_cleared", False)) and \
           bool(state.get("capacity_fresh_im", False)) and \
           (state.get("reason_code_im") in ALLOWED_IM)

def handoff_packet_complete(pkt: dict) -> bool:
    req = ["ct_head","vitals","labs","route_reason","ownership","capacity_check_ts"]
    return all(k in pkt for k in req) and pkt.get("ownership")=="trauma_until_im_accepts"

def dedupe_key(action: dict) -> str:
    return action.get("dedupe_key") or f"{action.get('type')}:{action.get('patient_id')}:{action.get('target_service','IM')}:{action.get('reason','NA')}"
'''
write_text(Path('ed_core')/'policy.py', policy_py)

bridge_py = r'''
class Bridge:
    def __init__(self):
        self.seen_ops = set()
    def apply(self, action: dict):
        op_id = action.get("op_id") or action.get("hash") or action.get("id")
        if op_id in self.seen_ops:
            return {"status":"skip_duplicate"}
        self.seen_ops.add(op_id)
        return {"status":"ok","routed":action.get("type"),"target":action.get("target_service","NA")}
'''
write_text(Path('ed_core')/'bridge.py', bridge_py)

py_compile.compile(str(Path('ed_core')/'policy.py'), doraise=True)
py_compile.compile(str(Path('ed_core')/'bridge.py'), doraise=True)
print("Policy & Bridge modules written and compiled.")


In [ ]:

# --- KPI replay (simple) ---
with open(Path('artifacts')/'thresholds_ops.json') as f:
    thr = json.load(f)
ACTIONS = ["NO_OP","ORDER_ECG","PERFORM_FAST","ORDER_LABS","ORDER_XR","ORDER_CT","REQUEST_CONSULT","REQUEST_BED"]

proposals = 0; approvals = 0; minutes_saved = []
for i in range(len(y_true_val)):
    top = int(np.argmax(probs_val[i]))
    top_name = ACTIONS[top]
    tau = float(thr.get(top_name, {}).get("tau", 0.5))
    if top_name!="NO_OP" and probs_val[i, top] >= tau:
        proposals += 1
        pmin = 0.7 if top_name in ("REQUEST_CONSULT","REQUEST_BED") else 0.5
        accept = probs_val[i, top] >= pmin
        approvals += int(accept)
        if accept:
            minutes_saved.append(4.0 + 6.0*np.random.rand())
if proposals==0:
    approve_rate = 0.0; false_page_rate = 0.0; med_ms = 0.0
else:
    approve_rate = approvals/proposals
    false_page_rate = (proposals-approvals)/proposals
    med_ms = float(np.median(minutes_saved)) if minutes_saved else 0.0

kpis = {"proposals": proposals, "approvals": approvals, "approve_rate": round(approve_rate,3),
        "false_page_rate": round(false_page_rate,3), "median_minutes_saved": round(med_ms,2)}
with open(Path('artifacts')/'kpi_report.json','w') as f:
    json.dump(kpis, f, indent=2)
print("KPIs:", kpis)


In [ ]:

# --- Audit anchoring (local) ---
from hashlib import sha256
from datetime import datetime

def merkle_root(records):
    if not records: return sha256(b"").hexdigest()
    layer = [sha256(r.encode()).hexdigest() for r in records]
    while len(layer)>1:
        nxt = []
        it = iter(layer)
        for a in it:
            b = next(it, a)
            nxt.append(sha256((a+b).encode()).hexdigest())
        layer = nxt
    return layer[0]

ACTIONS = ["NO_OP","ORDER_ECG","PERFORM_FAST","ORDER_LABS","ORDER_XR","ORDER_CT","REQUEST_CONSULT","REQUEST_BED"]
audit_lines = [f"proposal:{i}:{ACTIONS[int(np.argmax(probs_val[i]))]}:{float(np.max(probs_val[i])):.3f}" for i in range(min(50, len(y_true_val)))]
root = merkle_root(audit_lines)
stamp = {"root": root, "ts": datetime.utcnow().isoformat()+"Z"}
with open(Path('artifacts')/'audit_anchor.log','a') as f:
    f.write(json.dumps(stamp)+"\n")
print("Anchored audit root to artifacts/audit_anchor.log")
